In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
# %% [1] Environment Imports & Setup
import os
import shutil
import sys
import gymnasium as gym
import torch

sys.path.insert(0, os.path.abspath(".."))

from src.rl_2.env_adapter import MatchEnv
from src.rl_2.model import ActorCritic
from src.rl_2.pool import PoolOpponentController
from src.rl_2.ppo import train_mappo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


TEAM_SIZE = 2
PITCH_W_REG = 1200.0
PITCH_H_REG = 800.0
GOAL_H_REG = 220.0  
ROUND_STEPS_S2 = 3600 
NUM_ENVS = 24 

SAVE_DIR_S2 = "models/stage2/phase1"
POOL_DIR_S2 = os.path.join(SAVE_DIR_S2, "pool")
os.makedirs(POOL_DIR_S2, exist_ok=True)

Using device: cuda


In [3]:
# Seed Opponent Pool with Stage 1 Champion
STAGE1_BEST = "models/stage1/phase4/best_model.pt"
if os.path.exists(STAGE1_BEST):
  shutil.copy(STAGE1_BEST, os.path.join(POOL_DIR_S2, "champion.pt"))
  shutil.copy(STAGE1_BEST, os.path.join(POOL_DIR_S2, "history_0.pt"))
  print(f"✅ Initialized Stage 2 pool with Stage 1 Champion weights.")
else:
  raise FileNotFoundError(f"Missing Stage 1 model at: {STAGE1_BEST}")

def make_s2_env(env_rank: int):

  def _thunk():
    opp_ctrl = PoolOpponentController(
        pool_dir=POOL_DIR_S2,
        team="blue",
        device="cpu",
        p_random=0.05,
        p_heuristic=0.55,
    )
    env = MatchEnv(
        team_size=TEAM_SIZE,
        learner_team="red",
        max_round_steps=ROUND_STEPS_S2,
        goal_height=GOAL_H_REG,
        pitch_width=PITCH_W_REG,
        pitch_height=PITCH_H_REG,
        opponent_controller=opp_ctrl,
    )
    env.reset(seed=5000 + env_rank)
    return env

  return _thunk


envs_s2 = gym.vector.AsyncVectorEnv(
    [make_s2_env(i) for i in range(NUM_ENVS)],
    context="fork",
)


model_s2 = ActorCritic().to(device)
ckpt = torch.load(STAGE1_BEST, map_location=device, weights_only=False)
state_dict = (
    ckpt["model_state_dict"]
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt
    else ckpt
)
model_s2.load_state_dict(state_dict, strict=True)
print(f"🔥 Successfully warmstarted 2v2 Learner Model from: {STAGE1_BEST}")


train_mappo(
    envs=envs_s2,
    model=model_s2,
    device=device,
    team_size=TEAM_SIZE,
    total_timesteps=20_000_000,
    num_envs=NUM_ENVS,
    num_steps=256,  
    update_epochs=2, 
    minibatch_size=2048,
    lr_init=3e-5,  
    lr_final=3e-6,
    ent_coef_init=0.02,  
    ent_coef_final=0.006,
    gamma=0.996,
    gae_lambda=0.98,
    active_tiers=["heuristic"],
    target_tier="heuristic",
    filter_thresholds={
        "heuristic": 0.50,
    },
    goal_height=GOAL_H_REG,
    pitch_width=PITCH_W_REG,
    pitch_height=PITCH_H_REG,
    save_dir=SAVE_DIR_S2,
    pool_dir=POOL_DIR_S2,
    eval_episodes=50,  
    eval_freq=200_000,
    max_steps=ROUND_STEPS_S2,
)

envs_s2.close()

✅ Initialized Stage 2 pool with Stage 1 Champion weights.
🔥 Successfully warmstarted 2v2 Learner Model from: models/stage1/phase4/best_model.pt
🚀 MAPPO Initialized | Format: 2v2 | Envs: 24 | Step Batch: 12288 | Device: cuda

📊 [EVALUATION @ Step 208,896 | SPS: 4185 | Tiers: ['heuristic']]
   ⚔️  vs Heuristic [TARGET] | WR:  42.0% | Reward: +0.298 | Goals: 49 Scored, 35 Conceded (+14 Net)
   ❌ Retaining current baseline. Did not pass criteria for heuristic: [WR: None, Reward: None, Net: None]

📊 [EVALUATION @ Step 405,504 | SPS: 2794 | Tiers: ['heuristic']]
   ⚔️  vs Heuristic [TARGET] | WR:  50.0% | Reward: +0.762 | Goals: 50 Scored, 24 Conceded (+26 Net)
🏆 NEW CHAMPION REGISTERED @ step 405,504 -> models/stage2/phase1/pool/history_405504.pt
   ⭐⭐ PROMOTED! New Best Score (heuristic) -> [WR: 50.0%, Reward: +0.762, Net: +26]
      (Defeated previous record: [WR: None, Reward: None]) -> Saved: models/stage2/phase1/best_model.pt

📊 [EVALUATION @ Step 602,112 | SPS: 2493 | Tiers: ['heurist

In [10]:
import torch
from src.rl_2.visualization import evaluate_and_generate_html

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

evaluate_and_generate_html(
    red_agent="models/stage2/phase1/final_model.pt",
    blue_agent="models/stage2/phase1/final_model.pt",
    red_team_size=2,
    blue_team_size=2,
    device=device,
    filename="stage4_phase1_RLvsRL.html",
    num_episodes=10,
    max_steps=3600,
    base_seed=203,
)

🎬 Replay generated successfully: /home/minh-quan/Documents/Haxball project/training_2/render/stage4_phase1_RLvsRL.html


'render/stage4_phase1_RLvsRL.html'